In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from google.colab import drive
from sklearn.preprocessing import MinMaxScaler
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def process_and_merge_data_from_drive():
    folder_path = '/content/drive/MyDrive/Capstone Project/ASAH_COMPANY_CASE/'

    files = {
        'users': "users.xlsx",
        'completions': "developer_journey_completions.xlsx",
        'journeys': "developer_journeys.xlsx",
        'submissions': "developer_journey_submissions.xlsx",
        'exam_regs': "exam_registrations.xlsx",
        'exam_results': "exam_results.xlsx"
    }

    dfs = {}
    print("Membaca file dari Google Drive...")

    for key, filename in files.items():
        full_path = folder_path + filename
        try:
            dfs[key] = pd.read_excel(full_path)
            print(f"Berhasil membaca: {filename}")
        except FileNotFoundError:
            print(f"Error: File tidak ditemukan di {full_path}")
            return None
        except Exception as e:
            print(f"Gagal membaca file {filename}. Pesan error: {e}")
            return None

    # --- A. Rangkuman Submission (Tugas) ---
    print("Memproses data submission...")
    sub_agg = dfs['submissions'].groupby('submitter_id').agg({
        'id': 'count',
        'rating': 'mean'
    }).rename(columns={'id': 'total_submissions', 'rating': 'avg_submission_rating', 'submitter_id': 'user_id'})

    # --- B. Rangkuman Exam (Ujian) ---
    print("Memproses data ujian...")
    exam_merged = pd.merge(dfs['exam_results'], dfs['exam_regs'], left_on='exam_registration_id', right_on='id')
    exam_agg = exam_merged.groupby('examinees_id').agg({
        'score': 'mean',
        'is_passed': 'sum'
    }).rename(columns={'score': 'avg_exam_score', 'is_passed': 'total_exams_passed'})

    # --- C. Rangkuman Learning Journey (Kecepatan Belajar) ---
    print("Memproses data kecepatan belajar...")
    journey_info = dfs['journeys'][['id', 'hours_to_study']].rename(columns={'id': 'journey_id'})
    comp_merged = pd.merge(dfs['completions'], journey_info, on='journey_id', how='left')

    comp_merged['study_duration'] = comp_merged['study_duration'].replace(0, 1)
    comp_merged['speed_ratio'] = comp_merged['hours_to_study'] / comp_merged['study_duration']

    comp_agg = comp_merged.groupby('user_id').agg({
        'journey_id': 'count',
        'speed_ratio': 'mean'
    }).rename(columns={'journey_id': 'total_journeys_completed', 'speed_ratio': 'avg_speed_ratio'})

    # Merge and Clean
    print("Menggabungkan & Membersihkan data...")
    master_df = dfs['users'][['id', 'name', 'email', 'user_role']].rename(columns={'id': 'user_id'})

    master_df = master_df.merge(sub_agg, left_on='user_id', right_index=True, how='left')
    master_df = master_df.merge(exam_agg, left_on='user_id', right_index=True, how='left')
    master_df = master_df.merge(comp_agg, left_on='user_id', right_index=True, how='left')

    cols_zero = ['total_submissions', 'total_exams_passed', 'total_journeys_completed']
    master_df[cols_zero] = master_df[cols_zero].fillna(0)

    cols_mean = ['avg_submission_rating', 'avg_exam_score', 'avg_speed_ratio']
    for col in cols_mean:
        master_df[col] = master_df[col].fillna(master_df[col].mean())

    # Normalisasi
    scaler = MinMaxScaler()
    numeric_features = ['total_submissions', 'avg_submission_rating', 'avg_exam_score', 'total_journeys_completed', 'avg_speed_ratio']

    for col in numeric_features:
        master_df[f'norm_{col}'] = scaler.fit_transform(master_df[[col]])

    output_filename = folder_path + 'merged_normalized_user_data.csv'
    master_df.to_csv(output_filename, index=False)

    print(f"\nSelesai! File disimpan di Drive: {output_filename}")
    print(master_df.head())

    return master_df

df_result = process_and_merge_data_from_drive()

Membaca file dari Google Drive...
Berhasil membaca: users.xlsx
Berhasil membaca: developer_journey_completions.xlsx
Berhasil membaca: developer_journeys.xlsx
Berhasil membaca: developer_journey_submissions.xlsx
Berhasil membaca: exam_registrations.xlsx
Berhasil membaca: exam_results.xlsx
Memproses data submission...
Memproses data ujian...
Memproses data kecepatan belajar...
Menggabungkan & Membersihkan data...

Selesai! File disimpan di Drive: /content/drive/MyDrive/Capstone Project/ASAH_COMPANY_CASE/merged_normalized_user_data.csv
   user_id                    name                               email  \
0    96989        Inggih Wicaksono                  igihcksn@gmail.com   
1   938276  Nur Rizki Adi Prasetyo                 nrizki@dicoding.com   
2  5021477                  rifath              rifathali088@gmail.com   
3  5044844             LEDIS IDOLA  221113142@students.mikroskil.ac.id   
4  5051374        Fircan Ferdinand             kaslanafircan@gmail.com   

   user_role  to

In [ ]:
from google.colab import files
filename = 'final_data_normalized.csv'
df_result.to_csv(filename, index=False)
files.download(filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np
import json
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
import joblib

BASE_PATH = '/content/drive/MyDrive/Capstone Project/ASAH_COMPANY_CASE/'

# Fitur yang sudah dinormalisasi
features = [
    'norm_total_submissions',
    'norm_avg_submission_rating',
    'norm_avg_exam_score',
    'norm_total_journeys_completed',
    'norm_avg_speed_ratio'
]

# Cek apakah df_result ada
if 'df_result' not in locals():
    raise NameError("df_result tidak ditemukan! Harap jalankan kode 'Load & Process Data' terlebih dahulu.")

X = df_result[features]

# Training K-Means
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_result['cluster'] = kmeans.fit_predict(X)
centroids = df_result.groupby('cluster')[features].mean()
print("Rata-rata fitur per Cluster:")
print(centroids)

#Interpretasi
# Hitung rata-rata tiap cluster untuk melihat karakteristiknya
centroids = df_result.groupby('cluster')[features].mean()

# Penentuan label
fast_idx = int(centroids['norm_avg_speed_ratio'].idxmax())
reflective_idx = int(centroids.drop(fast_idx)['norm_avg_exam_score'].idxmax())
all_idxs = {0, 1, 2}
found_idxs = {fast_idx, reflective_idx}
consistent_idx = int(list(all_idxs - found_idxs)[0])

# Buat Mapping (Kamus)
label_map = {
    fast_idx: 'Fast Learner',
    reflective_idx: 'Reflective Learner',
    consistent_idx: 'Consistent Learner'
}

print(f"Mapping Terbentuk: {label_map}")

# Terapkan label ke DataFrame
df_result['learning_style'] = df_result['cluster'].map(label_map)

print("\nHasil Klasifikasi User:")
print(df_result[['name', 'learning_style', 'avg_exam_score', 'avg_speed_ratio']].head(10))

# Simpan CSV Hasil (Dengan Label)
csv_output = BASE_PATH + 'user_insights_labeled.csv'
df_result.to_csv(csv_output, index=False)
print(f"File CSV tersimpan di: {csv_output}")

# Scaler & JSON untuk front end
print("--- 3. Menyiapkan File JSON untuk Frontend ---")

# Scaler
original_features = [
    'total_submissions',
    'avg_submission_rating',
    'avg_exam_score',
    'total_journeys_completed',
    'avg_speed_ratio'
]

scaler = MinMaxScaler()
# Ambil data asli (bukan yang norm_) dari df_result
scaler.fit(df_result[original_features])

# Encoder
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NumpyEncoder, self).default(obj)

# DB Teks Rekomendasi
text_recommendations = {
    'Fast Learner': {
        'kategori': 'Fast Learner',
        'judul_saran': 'Tantangan Coding Time-Attack',
        'deskripsi_saran': 'Kamu memiliki kecepatan belajar tinggi! Hindari kebosanan dengan mengambil materi tingkat lanjut atau proyek deadline ketat.',
        'action_button': 'Mulai Tantangan Expert'
    },
    'Reflective Learner': {
        'kategori': 'Reflective Learner',
        'judul_saran': 'Analisis Studi Kasus Mendalam',
        'deskripsi_saran': 'Kamu tipe pemikir mendalam. Maksimalkan pemahamanmu dengan melakukan code review dan bedah studi kasus.',
        'action_button': 'Lihat Studi Kasus'
    },
    'Consistent Learner': {
        'kategori': 'Consistent Learner',
        'judul_saran': 'Rutin Belajar dengan Pomodoro',
        'deskripsi_saran': 'Disiplin adalah kekuatanmu. Jaga ritme belajar dengan teknik Pomodoro (25 menit fokus) agar tidak burnout.',
        'action_button': 'Set Jadwal Belajar'
    }
}

# Struktur JSON
model_export = {
    "metadata": {
        "version": "FINAL",
        "description": "Learning Style Model for Frontend Inference"
    },
    "features_order": original_features,
    "scaler": {
        "min": scaler.data_min_,
        "scale": scaler.data_range_
    },
    "centroids": kmeans.cluster_centers_,
    "cluster_map": label_map,
    "recommendation_texts": text_recommendations
}

# Simpan File JSON
json_output = BASE_PATH + 'model_recommendation_final.json'
with open(json_output, 'w') as f:
  json.dump(model_export, f, indent=4, cls=NumpyEncoder)

print(f"✅ SUKSES! File JSON tersimpan di: {json_output}")

# Simpan Scaler dan Model ke .h5
# Model
model_filename = BASE_PATH + 'model_learning_style.h5'
joblib.dump(kmeans, model_filename)
print(f"Model KMeans berhasil disimpan di: {model_filename}")

#Scaler
scaler_filename = BASE_PATH + 'scaler_learning_style.h5'
joblib.dump(scaler, scaler_filename)
print(f"Scaler berhasil disimpan di: {scaler_filename}")


Rata-rata fitur per Cluster:
         norm_total_submissions  norm_avg_submission_rating  \
cluster                                                       
0                      0.075126                    0.411057   
1                      0.204830                    0.620679   
2                      0.826705                    0.091826   

         norm_avg_exam_score  norm_total_journeys_completed  \
cluster                                                       
0                   0.514454                       0.200542   
1                   0.692054                       0.459756   
2                   0.137204                       0.792683   

         norm_avg_speed_ratio  
cluster                        
0                    0.739559  
1                    0.432988  
2                    0.794315  
Mapping Terbentuk: {2: 'Fast Learner', 1: 'Reflective Learner', 0: 'Consistent Learner'}

Hasil Klasifikasi User:
                     name      learning_style  avg_exam_score  av

In [ ]:
from google.colab import files
files.download(scaler_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>